In [1]:
# Celda 1
%cd C:/Users/fabi_/Documents/workflow/textos-catolicos-xix

import csv
import os
import re
import statistics
from difflib import SequenceMatcher  # Standard library, measures how alike two strings are

# To test on the pilot sample: OCR_FOLDER = "ocr-piloto/textos-ocr-muestra"
# and CLEAN_FOLDER = "ocr-piloto/textos-limpios-muestra":
OCR_FOLDER = "textos-ocr"         # CSVs of lines, from 4-ocr-corpus
CLEAN_FOLDER = "textos-limpios"   # One .txt per book

# Books whose small type at the foot of the page is the author's and not the editor's: only footnotes with a
# marker go (Celda 4, step 2 is skipped). Ágreda, Mística Ciudad de Dios: chapter summaries and notes are already
# in the first edition (1670), so they stay:
MARKERS_ONLY_BOOKS = ("agreda-mistica-",)

# Thresholds, tested on the pilot sample (15 books x 5 pages). Positions are fractions of the page:
SMALL = 0.88         # A line is small type if its height (or letter width) is under 88% of the body text's
FOOT = 0.85          # The foot of the page: its bottom 15%
NOTES_FROM = 0.4     # A footnote starts below 40% of the page
HEAD_ZONE = 0.2      # Headers sit in the top 20% of the page
LEFT_COLUMN = 0.42   # A line starting left of 42% of the page width belongs to the left column
NEIGHBORS = 4        # A header is compared with the top lines of up to 4 pages before and after
SIMILARITY = 0.8     # 0 = nothing alike, 1 = identical
TALL = 2.5           # A line 2.5 times the body height or more is an ornament, unless it is made of real words
KNOWN_IN_BOOKS = 10  # A word is real if it appears in 10 books or more of the OCR

C:\Users\fabi_\Documents\workflow\textos-catolicos-xix


In [2]:
# Celda 2
# Read one book's CSV: {page number: [line, line, ...]} and {page number: (width, height)}.
# Lines stay in Tesseract's reading order:
def read_book(path):
    pages = {}
    sizes = {}
    with open(path, encoding="utf-8", newline="") as f:
        for row in csv.DictReader(f):
            number = int(row["page"])
            line = {"text": row["text"], "height": float(row["height"])}
            for key in ["left", "top", "right", "bottom"]:
                line[key] = int(row[key])
            pages.setdefault(number, []).append(line)
            sizes[number] = (int(row["page_width"]), int(row["page_height"]))
    return pages, sizes

In [3]:
# Celda 3
# Type size. A line's height is not enough: OCR sometimes measures it badly, so letter width counts too.
def letter_width(line):
    return (line["right"] - line["left"]) / max(len(line["text"]), 1)

# The body text's size is measured over the whole book, since on some pages notes take up half the page.
# The 80th percentile lands on body text as long as notes are under 80% of the lines:
def percentile_80(values):
    values = sorted(values)
    return values[int(len(values) * 0.8)]

def body_size(all_lines):
    body_height = percentile_80([line["height"] for line in all_lines])
    body_width = percentile_80([letter_width(line) for line in all_lines if len(line["text"]) >= 12])
    return body_height, body_width

# Small on either measure (used for footnotes):
def is_small(line, body, limit=SMALL):
    body_height, body_width = body
    if line["height"] / body_height < limit:
        return True
    return len(line["text"]) >= 12 and letter_width(line) / body_width < limit - 0.03

# Small on both measures (used where one bad measure is not enough: <small> blocks, margin notes):
def is_small_strict(line, body):
    body_height, body_width = body
    if line["height"] / body_height >= SMALL:
        return False
    return len(line["text"]) < 12 or letter_width(line) / body_width < SMALL

In [4]:
# Celda 4
# Footnotes. Note markers, with common OCR misreadings:
# "(1)", "(11)", "(a)", "110)" for "(10)", "\9)", "(5.", "1 Eccles.", "* ":
MARKER = re.compile(r"^([\(\[\\|]?\s*\w{1,3}\s*[\)\]]|\(\s*\w{1,3}\s*[.,;]|\d{1,3}\s+[A-Z]|[*†‡]\s)")

# A line at the foot of the page holding several notes, "(a) Psalm. 25. (b) 1. Cor. 11. (c) Tract. 26.": its wide
# spacing and parentheses keep it from being measured small, so it counts as notes by its shape:
def several_notes(line, page_height):
    text = line["text"]
    return (line["top"] >= page_height * FOOT and bool(MARKER.match(text))
            and len(re.findall(r"\(\s*\w{1,2}\s*\)", text)) >= 2)

# On two-column pages each column has its own notes. A page is taken as one column if fewer than 5 lines start on the right:
def columns(lines, page_width):
    left = [line for line in lines if line["left"] < page_width * LEFT_COLUMN]
    right = [line for line in lines if line["left"] >= page_width * LEFT_COLUMN]
    if len(right) < 5:
        return [sorted(lines, key=lambda line: line["top"])]
    return [sorted(left, key=lambda line: line["top"]), sorted(right, key=lambda line: line["top"])]

# Returns the ids of the footnote lines of one page. With markers_only, step 2 is skipped:
def find_notes(lines, page_width, page_height, body, markers_only=False):
    notes = set()
    for column in columns(lines, page_width):
        gaps = [column[k + 1]["top"] - column[k]["top"] for k in range(len(column) - 1)]
        line_gap = statistics.median(gaps) if gaps else 0
        # 1. The first small line below 40% of the page that starts with a marker.
        #    At the foot of the page it only needs to be a bit smaller than the body.
        #    Every long line (30 characters or more) from there to the bottom must be small too: a misread subtitle
        #    ("De) sacramento de..." for "Del sacramento de...") or a stray "*" looks like a marker,
        #    but is followed by body text. Then the next marker is tried:
        start = None
        for k in range(len(column)):
            line = column[k]
            limit = 0.95 if line["top"] >= page_height * FOOT else SMALL
            marked = MARKER.match(line["text"]) and is_small(line, body, limit)
            if line["top"] >= page_height * NOTES_FROM and (marked or several_notes(line, page_height)):
                body_below = [other for other in column[k:] if len(other["text"]) >= 30
                              and not is_small(other, body, 0.95) and not several_notes(other, page_height)]
                if len(body_below) == 0:
                    start = k
                    break
        # 2. No marker (e.g. a superscript the OCR missed): a run of small lines that closes the column at the foot of the page.
        #    At least one of them must be long enough (12 characters) to measure its letter width too: a short line
        #    without tall letters ("con vos.»", ending a paragraph) is measured small by height alone:
        if start is None and not markers_only:
            k = len(column)
            while k > 0 and column[k - 1]["top"] >= page_height * FOOT and is_small_strict(column[k - 1], body):
                k = k - 1
            if k < len(column) and any(len(line["text"]) >= 12 for line in column[k:]):
                start = k
        if start is None:
            continue
        # 3. A note carried over from the previous page sits just above the first marker, without a marker.
        #    A wider blank than the usual line spacing means a different block (e.g. a chapter summary), so stop there.
        #    A short line (under 12 characters) ending a paragraph of the note is measured by height alone, which is
        #    unreliable, so it only needs to be a bit smaller than the body ("Teresa.", in Teresa, Poesias, p. 2):
        while (start > 0 and (is_small(column[start - 1], body)
                              or (len(column[start - 1]["text"]) < 12 and is_small(column[start - 1], body, 0.95)))
               and column[start]["top"] - column[start - 1]["top"] <= 1.2 * line_gap):
            start = start - 1
        # 4. A block of small type right under a heading is part of the work, not a note. is_capitals is defined in Celda 6:
        if start > 0 and is_capitals(column[start - 1]["text"]):
            continue
        # Everything from there to the bottom of the column is notes:
        for line in column[start:]:
            notes.add(id(line))
    return notes

# Margin notes (Ochoa, Tesoro): a short line shaped like a reference, with common OCR misreadings:
# "Galat., 3.", "Rom,, 8.", "Psal.. 18.", "Josué, 10.", "Luc., $." ($ for 5), "1. Cor., 9.", "Ad Rom., cap. 8.",
# "Joan., 1, cap. 3.". The book name has 2 letters or more and is not in capitals, so "CAPITULO XI." and "§ I"
# (read "s. L") are not taken:
CITATION = re.compile(r"^\W*(\d[.,]?\s*)?((Ad|San)\s+)?[A-ZÀ-Ýa-zà-ÿ][a-zà-ÿ]{1,7}\s?[.,']{1,3}\s*(\d{1,3}\s*,\s*)?"
                      r"(cap[.,:]*\s*)?[\dIVXLivxl%$]{1,4}\W*([\dIVXLivxl%$]{1,4}\W*)?$")
# Some margin citations are not measured as small. Those still count if the name is followed by a comma, as in
# Ochoa ("Psal., 129."), since a citation inside the author's text is written without it ("Apoca-" / "lyp. vi."):
NAME_AND_COMMA = re.compile(r"^\D*[a-zà-ÿ][.']*,")
# "San Gregor.", "San Agust.": only when small on both measures, since "San Pablo." can end a line of body text:
SAINT = re.compile(r"^\W*San\s+[A-Z][a-zà-ÿ]{2,8}\.\W*$")

def find_margin_notes(lines, body):
    margin = set()
    for line in lines:
        text = line["text"]
        small = is_small_strict(line, body)
        if len(text) <= 25 and CITATION.match(text) and (small or NAME_AND_COMMA.match(text)):
            margin.add(id(line))
        elif len(text) <= 25 and SAINT.match(text) and small:
            margin.add(id(line))
    return margin

In [5]:
# Celda 5
# Headers: lines at the top of the page, removed from the top down while they are junk,
# a page number, a short line with a number at either end, or repeated on nearby pages.
# "143 VIDA DE SANTA TERESA." -> "vida de santa teresa":
def normalize(text):
    return re.sub(r"[\W\d_]+", " ", text.lower()).strip()

# Fewer than 4 letters, or less than half letters: "* 4", "a ¿ -", "T. X.":
def is_junk(text):
    letters = len(re.findall(r"[^\W\d_]", text))
    return letters < 4 or letters / max(len(text.replace(" ", "")), 1) < 0.5

# "143", "l43" (OCR for 143), "- 12 -", "xiv" (lowercase Roman, front matter):
def is_page_number(text):
    return bool(re.fullmatch(r"[\W_]*([0-9lIO%]*[0-9][0-9lIO%]*|[ivxlc]+)[\W_]*", text.strip()))

# A short line that starts or ends with a number: "Doctrina cristiana 117", "116 Declaracion de la":
def has_edge_number(text):
    return len(text) <= 50 and bool(re.search(r"^\W*\d{1,4}\b|\b\d{1,4}\W*$", text))

# Like one of the first 3 lines of the 4 pages before or after (headers alternate between even and odd pages):
def is_repeated(text, i, top_texts):
    a = normalize(text)
    if len(a) < 3:
        return False
    for j in range(i - NEIGHBORS, i + NEIGHBORS + 1):
        if j != i and 0 <= j < len(top_texts):
            for other in top_texts[j]:
                b = normalize(other)
                if len(b) >= 3 and SequenceMatcher(None, a, b).ratio() >= SIMILARITY:
                    return True
    return False

# Returns the ids of the header lines of page i:
def find_header(lines, i, top_texts, page_height):
    header = set()
    for line in sorted(lines, key=lambda line: line["top"]):
        text = line["text"]
        if line["top"] > page_height * HEAD_ZONE:
            break
        if is_junk(text) or is_page_number(text) or has_edge_number(text) or is_repeated(text, i, top_texts):
            header.add(id(line))
        else:
            break
    return header

# Leftovers at the foot of the page: "6*", "5d", "T. X.", "8" (no two letters in a row):
def is_foot_junk(line, page_height):
    return line["top"] >= page_height * FOOT and not re.search(r"[^\W\d_]{2}", line["text"])

# Library stamps on the scans: "© Biblioteca Nacional de España" (read "O Biblioteca..."), "Digitized by Google", "UNIVERSITY OF MINNESOTA":
WATERMARK = re.compile(r"^\W{0,3}[OC©]?\s*Biblioteca\s+Nacional(\s+de\s+Espa\w+)?\W*$|^\W*(Digiti[sz]ed\s+by\s*)?Google\W*$|^\W*University\s+of\s+Minnesota\W*$",
                       re.IGNORECASE)

def is_watermark(line):
    return bool(WATERMARK.match(line["text"].strip()))

# Printer's signatures: a line at the foot of the page, with a number or a Roman numeral, repeated (identical once
# normalized) on 3 pages or more of the book: "VARONES ILUSTRES.—TOMO VII 14" every 16 pages, "PARTE I.".
# "FIN DEL TOMO VII" appears once, so it stays; "Señor." ending a paragraph has no numeral, so it stays too:
NUMERAL = re.compile(r"\d|\b[IVXL]+\b")

def repeated_feet(pages, sizes):
    counts = {}
    for n in pages:
        page_height = sizes[n][1]
        feet = set()
        for line in pages[n]:
            if line["top"] >= page_height * FOOT and NUMERAL.search(line["text"]):
                feet.add(normalize(line["text"]))
        for text in feet:
            if len(text) >= 5:
                counts[text] = counts.get(text, 0) + 1
    return set(text for text in counts if counts[text] >= 3)

# Compared by similarity, since the OCR misreads the signature number: "VARONES ILUSTRES.—TOMO VII a":
def is_signature(line, page_height, feet):
    if line["top"] < page_height * FOOT:
        return False
    text = normalize(line["text"])
    for foot in feet:
        if SequenceMatcher(None, text, foot).ratio() >= SIMILARITY:
            return True
    return False

In [6]:
# Celda 6
# Marks: paratexts that cannot be told apart from the author's text with certainty are marked, not removed.
# <head>: a line all in capitals ("CAPÍTULO XXI.", "FIN DEL TOMO PRIMERO").
# <small>: a run of 2 or more small-type lines (chapter summaries, verses, Latin quotations).
def is_capitals(text):
    letters = re.findall(r"[^\W\d_]", text)
    return len(letters) >= 3 and all(letter.isupper() for letter in letters)

def mark(lines, body):
    texts = []
    k = 0
    while k < len(lines):
        end = k
        while end < len(lines) and is_small_strict(lines[end], body):
            end = end + 1
        if end - k >= 2:
            block = [line["text"] for line in lines[k:end]]
            block[0] = "<small>" + block[0]
            block[-1] = block[-1] + "</small>"
            texts.extend(block)
            k = end
        else:
            text = lines[k]["text"]
            if is_capitals(text):
                text = "<head>" + text + "</head>"
            texts.append(text)
            k = k + 1
    return texts

In [7]:
# Celda 7
# Words split at the end of a line or a page: "el pe-" + "cado de" -> "el pecado" + "de".
# Only a hyphen after a letter counts. The OCR reads the hyphen in several ways: "-", "=" (the double hyphen of
# old prints), "—", "–", "¬" or "»". Some prints repeat it at the start of the next line ("disi-" / "- mular"),
# so a hyphen there is dropped too, and so is the quotation mark that opens every line of a quotation in some prints
# ("aun-" / "» que" -> "aunque", "es-" / "«to" -> "esto").
SURE_HYPHENS = "-="        # In the corpus the hyphen has no other use at the end of a line: always a split word
DOUBTFUL_HYPHENS = "—–¬»"  # Also a dash or a closing quotation mark ("Señora»"), so only before a lowercase letter
ALL_HYPHENS = SURE_HYPHENS + DOUBTFUL_HYPHENS

# "el pe-" -> "el pe", "male—" -> "male", "Por-=" -> "Por". None if the line does not end in a hyphen after a letter:
def hyphen_stem(text):
    stem = text.rstrip(ALL_HYPHENS)
    if stem == text or stem == "" or not stem[-1].isalpha():
        return None
    return stem

# "- mular cosa alguna." -> ("mular", "cosa alguna."):
def first_word(lower):
    words = lower.strip().lstrip("-=—–»«“\" ").split(" ", 1)
    if len(words) > 1:
        return words[0], words[1]
    return words[0], ""

# A sure hyphen always joins; a doubtful one only if the lower line goes on in lowercase. A closing quotation mark
# before a lowercase word is joined by mistake ("estudiantes» y" -> "estudiantesy"); that is the price of a simple rule.
# Never into a <head> line. Into a <small> block only if the line has more words, so the tag stays:
# "san-" + "<small>tidad de" -> "santidad" + "<small>de":
def should_join(upper, lower):
    stem = hyphen_stem(upper)
    if stem is None or lower.strip() == "" or lower.startswith("<head>"):
        return False
    first, rest = first_word(lower.removeprefix("<small>"))
    if first == "" or first.startswith("<") or (lower.startswith("<small>") and rest == ""):
        return False
    if upper[len(stem)] in SURE_HYPHENS:
        return True
    return first[0].islower()

# ("el pe-", "cado de") -> ("el pecado", "de"):
def join_two_lines(upper, lower):
    tag = "<small>" if lower.startswith("<small>") else ""
    first, rest = first_word(lower.removeprefix("<small>"))
    if rest == "":
        return hyphen_stem(upper) + first, ""
    return hyphen_stem(upper) + first, tag + rest

# A word never goes on in a <head> line: a heading there (a header or signature that was not removed,
# "<head>PUNTO</head>", or capitals misread by the OCR) is skipped, so the word joins the text after it:
def next_text(texts, start):
    j = start
    while j < len(texts) and texts[j].startswith("<head>"):
        j = j + 1
    return j

def join_hyphens(pages):
    # Inside each page:
    for texts in pages:
        k = 0
        while k < len(texts) - 1:
            j = next_text(texts, k + 1)
            if j < len(texts) and should_join(texts[k], texts[j]):
                texts[k], texts[j] = join_two_lines(texts[k], texts[j])
                if texts[j] == "":
                    texts.pop(j)  # The lower line was only the end of the word
            k = k + 1
    # Across pages (the whole word stays on the earlier page, before the <pb/>):
    for i in range(len(pages) - 1):
        j = next_text(pages[i + 1], 0)
        if len(pages[i]) > 0 and j < len(pages[i + 1]) and should_join(pages[i][-1], pages[i + 1][j]):
            pages[i][-1], pages[i + 1][j] = join_two_lines(pages[i][-1], pages[i + 1][j])
            if pages[i + 1][j] == "":
                pages[i + 1].pop(j)

In [8]:
# Celda 8
# Quotation marks at the start of every line of a quotation, as many old prints have them:
#   «lo puedan hacer, procuramos defenderlos de toda
#   «injuria y calumnia, y de conservarlos enteros sin
# Today only the opening and the closing marks are printed. So in a run of 2 or more lines that start with a quotation
# mark, the mark stays only on the first line, where it opens the quotation, and goes from the others. The first line
# loses it too when it is "»" (in Spanish it never opens a quotation; some prints use it to carry one on) or when the
# line before already opened the quotation ("...dice el Papa: «Nos que"). Closing marks, at the end of a line, stay.
LINE_QUOTE = re.compile(r"(<small>)?\s*[«»\"“”„]\s*")

def leaves_quote_open(text):
    return text.count("«") > text.count("»") or text.count("“") > text.count("”")

def remove_line_quotes(pages):
    places = [(i, k) for i in range(len(pages)) for k in range(len(pages[i]))]  # every line of the book, in order
    start = 0
    while start < len(places):
        end = start
        while end < len(places) and LINE_QUOTE.match(pages[places[end][0]][places[end][1]]):
            end = end + 1
        if end - start >= 2:
            for r in range(start, end):
                i, k = places[r]
                text = pages[i][k]
                match = LINE_QUOTE.match(text)
                if r == start and "»" not in match.group(0):
                    before = pages[places[start - 1][0]][places[start - 1][1]] if start > 0 else ""
                    if not leaves_quote_open(before):
                        continue  # The opening mark stays
                pages[i][k] = (match.group(1) or "") + text[match.end():]
            start = end
        else:
            start = max(end, start + 1)

In [9]:
# Celda 9
# Ornaments and drop caps, both read by the OCR as letters.
# Real words: those in 10 books or more of the OCR. About a minute:
WORD = re.compile(r"[^\W\d_]+")

def count_ocr_books(folder):
    books = {}
    for file_name in os.listdir(folder):
        if file_name.endswith(".csv"):
            words = set()
            with open(os.path.join(folder, file_name), encoding="utf-8", newline="") as f:
                for row in csv.DictReader(f):
                    for word in WORD.findall(row["text"]):
                        words.add(word.lower())
            for word in words:
                books[word] = books.get(word, 0) + 1
    return books

ocr_books = count_ocr_books(OCR_FOLDER)

# Share of real words among those of 3 letters or more: "LIBRO TERCERO" -> 1.0, "PEGIESDEGINCINCORS" -> 0.0:
def real_share(text):
    words = [word for word in WORD.findall(text) if len(word) >= 3]
    if len(words) == 0:
        return 0.0
    real = [word for word in words if ocr_books.get(word.lower(), 0) >= KNOWN_IN_BOOKS]
    return len(real) / len(words)

# Ornaments (vignettes, rules, specks at the edge of the scan). Two shapes, both without real words:
# - very tall: "PEGIESDEGINCINCORS" (a vignette, 3.4 times the body), "A" (a speck the height of the page);
# - mostly lone capitals, 5 tokens or more: "SA O O E O O JA do O E GS".
def is_ornament(line, body):
    text = line["text"]
    if real_share(text) >= 0.5:
        return False
    if line["height"] >= TALL * body[0]:
        return True
    tokens = text.split()
    capitals = [token for token in tokens if len(token) <= 2 and token.isalpha() and token.isupper()]
    return len(tokens) >= 5 and len(capitals) / len(tokens) >= 0.6

# Drop caps. The OCR reads the large initial as the start of the next line, at the same height and further left:
#   "E los dos libros pasados..."   (left 170)        "De los dos libros pasados..."
#   "D en ellos lo que..."          (left 86)    ->   "en ellos lo que..."
# The rest of the first word is often in capitals ("ICHO hemos" + "D ro del" -> "Dicho hemos" + "ro del"):
DROP_CAP = re.compile(r"^([A-ZÁÉÍÓÚÑ]) (.*)$")
FIRST_CAPITALS = re.compile(r"^([A-ZÁÉÍÓÚÑ]{1,6})\b(.*)$")

def fix_drop_caps(lines, body):
    body_height, body_width = body
    for k in range(len(lines) - 1):
        upper, lower = lines[k], lines[k + 1]
        cap = DROP_CAP.match(lower["text"])
        first = FIRST_CAPITALS.match(upper["text"])
        if cap is None or first is None:
            continue
        has_lowercase = re.search(r"[a-záéíóúñ]", upper["text"])  # Not a heading: "CARTA CXXII." + "A la madre..."
        same_height = abs(upper["top"] - lower["top"]) <= body_height
        further_left = upper["left"] - lower["left"] >= 2 * body_width
        if has_lowercase and same_height and further_left:
            word = cap.group(1) + first.group(1)
            upper["text"] = word.capitalize() + first.group(2)
            lower["text"] = cap.group(2)


In [10]:
# Celda 10
# Clean every book: remove ornaments, footnotes, margin notes, headers, foot junk, library stamps and signatures; fix drop caps; mark <head> and <small>; remove the quotation marks
# that start every line of a quotation; join hyphens.
# Each page starts with <pb n="N"/>, N being the page of the PDF. Unlike the OCR, this cell is fast and can be rerun at will:
os.makedirs(CLEAN_FOLDER, exist_ok=True)
for file_name in sorted(os.listdir(OCR_FOLDER)):
    if not file_name.endswith(".csv"):
        continue  # Skips a ".csv.part" left by a stopped OCR
    pages, sizes = read_book(os.path.join(OCR_FOLDER, file_name))
    numbers = sorted(pages)
    body = body_size([line for n in numbers for line in pages[n]])
    top_texts = []
    for n in numbers:
        top_lines = sorted(pages[n], key=lambda line: line["top"])[:3]
        top_texts.append([line["text"] for line in top_lines])
    feet = repeated_feet(pages, sizes)

    clean_pages = []
    removed = 0
    for i in range(len(numbers)):
        lines = [line for line in pages[numbers[i]] if not is_watermark(line) and not is_ornament(line, body)]
        fix_drop_caps(lines, body)
        page_width, page_height = sizes[numbers[i]]
        drop = find_notes(lines, page_width, page_height, body, file_name.startswith(MARKERS_ONLY_BOOKS))
        drop = drop | find_margin_notes(lines, body)
        drop = drop | find_header(lines, i, top_texts, page_height)
        kept = []
        for line in lines:
            if id(line) not in drop and not is_foot_junk(line, page_height) and not is_signature(line, page_height, feet):
                kept.append(line)
        removed = removed + len(pages[numbers[i]]) - len(kept)
        clean_pages.append(mark(kept, body))

    remove_line_quotes(clean_pages)
    join_hyphens(clean_pages)

    parts = []
    for n, texts in zip(numbers, clean_pages):
        parts.append('<pb n="' + str(n) + '"/>\n' + "\n".join(texts))
    with open(os.path.join(CLEAN_FOLDER, file_name[:-4] + ".txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(parts))
    print(file_name, len(numbers), "pages,", removed, "lines removed")

agreda-cartas-a-felipe-iv-tomo1.csv 407 pages, 695 lines removed


agreda-cartas-a-felipe-iv-tomo2.csv 740 pages, 1190 lines removed


agreda-mistica-ciudad-de-dios-tomo1.csv 40 pages, 69 lines removed


agreda-mistica-ciudad-de-dios-tomo2.csv 303 pages, 717 lines removed


agreda-mistica-ciudad-de-dios-tomo3.csv 369 pages, 997 lines removed


agreda-mistica-ciudad-de-dios-tomo4.csv 402 pages, 1142 lines removed


... (121 outputs left out)

vegas-poesia-cristiana.csv 91 pages, 243 lines removed


venegas-agonia-del-transito-de-la-muerte.csv 208 pages, 518 lines removed


yepes-vida-de-santa-teresa-de-jesus.csv 315 pages, 470 lines removed


zarate-discursos-de-la-paciencia-cristiana.csv 264 pages, 545 lines removed


In [11]:
# Celda 11
# Words split by a space the OCR put inside them: "imper fecto" -> "imperfecto", "J esucristo" -> "Jesucristo".
# It runs over the clean texts of the whole corpus, after Celda 10, because it needs the corpus vocabulary.
# A word "exists" if it appears in 10 books or more, so that an OCR error repeated in one book does not count
# Words are compared in lowercase and without accents.
# Two pieces are joined only if one of them does not exist and the joined word does: "hallas en" stays.
# The same goes for a word split at the end of a line without a hyphen (the OCR lost it): "la verda" / "dera doctrina"
# -> "la verdadera" / "doctrina". The second piece must start in lowercase; page breaks are left alone.
# Never with a lone "á", "é", "ó" or "ú" (compared without accents, "remití á" would become "remitía"),
# nor with a capital inside ("en Salces"), except a capital split from its word ("A póstoles").
import unicodedata

MIN_BOOKS = 10
LONE_ACCENTED = ["á", "é", "ó", "ú", "Á", "É", "Ó", "Ú"]
WORD = re.compile(r"[^\W\d_]+")
PAIR = re.compile(r"(?<![^\W\d_])([^\W\d_]+) ([^\W\d_]+)(?![^\W\d_])")

# "Apóstoles" -> "apostoles":
def compare_key(word):
    letters = unicodedata.normalize("NFD", word.lower())
    return "".join(c for c in letters if unicodedata.category(c) != "Mn")

# {word: number of books where it appears}:
def count_books(folder):
    books = {}
    for file_name in os.listdir(folder):
        if file_name.endswith(".txt"):
            with open(os.path.join(folder, file_name), encoding="utf-8") as f:
                text = re.sub(r"<[^>]+>", " ", f.read())
            for key in set(compare_key(word) for word in WORD.findall(text)):
                books[key] = books.get(key, 0) + 1
    return books

def join_split_words(line, books):
    def exists(word):
        return books.get(compare_key(word), 0) >= MIN_BOOKS

    def join(match):
        a, b = match.group(1), match.group(2)
        if a in LONE_ACCENTED or b in LONE_ACCENTED:
            return match.group(0)
        if b[0].isupper() and not (len(a) == 1 and a.isupper()):
            return match.group(0)
        if (exists(a) and exists(b)) or not exists(a + b):
            return match.group(0)
        return a + b

    return PAIR.sub(join, line)

LAST_WORD = re.compile(r"(?<![^\W\d_])([^\W\d_]+)$")
FIRST_WORD = re.compile(r"^([^\W\d_]+)(?![^\W\d_])")

# "la verda" + "dera doctrina" -> "la verdadera" + "doctrina"; None if the two pieces are not joined:
def join_line_end(line, next_line, books):
    last = LAST_WORD.search(line)
    first = FIRST_WORD.match(next_line)
    if last is None or first is None:
        return None
    a, b = last.group(1), first.group(1)
    if not b[0].islower() or a in LONE_ACCENTED or b in LONE_ACCENTED:
        return None
    a_exists = books.get(compare_key(a), 0) >= MIN_BOOKS
    b_exists = books.get(compare_key(b), 0) >= MIN_BOOKS
    if (a_exists and b_exists) or books.get(compare_key(a + b), 0) < MIN_BOOKS:
        return None
    return line + b, next_line[len(b):].lstrip()

books = count_books(CLEAN_FOLDER)
total = 0
across = 0
for file_name in sorted(os.listdir(CLEAN_FOLDER)):
    path = os.path.join(CLEAN_FOLDER, file_name)
    with open(path, encoding="utf-8") as f:
        lines = f.read().split("\n")
    joined = 0
    for k in range(len(lines)):
        if not lines[k].startswith("<pb"):
            new = join_split_words(lines[k], books)
            joined = joined + len(lines[k].split(" ")) - len(new.split(" "))
            lines[k] = new
    for k in range(len(lines) - 1):
        if not lines[k].startswith("<pb") and not lines[k + 1].startswith("<pb"):
            pair = join_line_end(lines[k], lines[k + 1], books)
            if pair is not None:
                lines[k], lines[k + 1] = pair
                across = across + 1
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    total = total + joined
print(total, "words joined inside a line,", across, "across a line end")

238 words joined inside a line, 2224 across a line end


In [12]:
# Celda 12
# Stray punctuation, specks of the scan read as dots, colons, commas or apostrophes.
# It runs over the clean texts after Celda 11, because it needs to know which words are abbreviations and which words exist.
# 1. An apostrophe inside a word: joined if the joined word exists and the pieces do not, as in Celda 11
#    ("conside'racion" -> "consideracion"); otherwise it becomes a space ("admitió'sin" -> "admitió sin").
#    An apostrophe at the start or end of a word goes ("luego 'prosiguió" -> "luego prosiguió", "llegó' primero" -> "llegó primero").
# 2. A comma between two letters gets a space: "morir,hizo" -> "morir, hizo".
# 2b. A quotation mark between two letters becomes a space: 'tres"maneras' -> "tres maneras", "se“ocupaba" -> "se ocupaba".
# 3. A dot or colon between two words, with no space: "en.la" -> "en la", "miraba:su" -> "miraba su".
#    Both words need 2 letters or more and the second one starts in lowercase, so "v.gr." and "S.Pablo" stay.
#    Repeated until nothing changes, for chains: "de.la.casa" -> "de la casa".
# 4. A dot followed by a lowercase word: "de una. tierra" -> "de una tierra". The dot stays after an abbreviation:
#    a word written with a dot in 20% of its appearances or more in the corpus ("cap. primero", "etc. y").
#    Sometimes the dot was a comma misread ("norte. como"); the comma is lost either way.
# 5. Dots and colons at the start of a word go: "pues :el Señor" -> "pues el Señor", "/.la tierra" -> "/la tierra".
#    If the word before is an abbreviation without its dot, the dot moves to it: "S .Pablo" -> "S. Pablo".
LETTER = r"[^\W\d_]"
APOSTROPHE_INSIDE = re.compile(r"(?<![^\W\d_])([^\W\d_]+)['’]([^\W\d_]+)(?![^\W\d_])")
APOSTROPHE_AT_EDGE = re.compile(r"(?<![^\W\d_])['’](?=[^\W\d_])|(?<=[^\W\d_])['’](?![^\W\d_])")
COMMA_INSIDE = re.compile(r"(?<=[^\W\d_]),(?=[^\W\d_])")
QUOTE_INSIDE = re.compile(r"(?<=[^\W\d_])[\"“”](?=[^\W\d_])")
INSIDE = re.compile(r"(?<![^\W\d_])([^\W\d_]{2,})[.:]([a-záéíóúñ][^\W\d_]+)(?![^\W\d_])")
DOT_THEN_LOWERCASE = re.compile(r"(?<![^\W\d_])([^\W\d_]+)\. (?=[a-záéíóúñ])")
DOTS_AT_START = re.compile(r"(?:(?<![^\W\d_])([^\W\d_]+) )?(?<!\S)[.:]+(?=[^\W\d_])")
WORD_AND_DOT = re.compile(r"(?<![^\W\d_])([^\W\d_]+)(\.?)")
ABBREVIATION_SHARE = 0.2

# {word: (appearances, appearances followed by a dot)}, in lowercase:
def count_dots(folder):
    counts = {}
    for file_name in os.listdir(folder):
        if file_name.endswith(".txt"):
            with open(os.path.join(folder, file_name), encoding="utf-8") as f:
                text = re.sub(r"<[^>]+>", " ", f.read())
            for word, dot in WORD_AND_DOT.findall(text):
                total, dotted = counts.get(word.lower(), (0, 0))
                counts[word.lower()] = (total + 1, dotted + (dot == "."))
    return counts

def is_abbreviation(word, counts):
    total, dotted = counts.get(word.lower(), (1, 0))
    return dotted / total >= ABBREVIATION_SHARE

def fix_apostrophes(line, books):
    def exists(word):
        return books.get(compare_key(word), 0) >= MIN_BOOKS

    def join_or_split(match):
        a, b = match.group(1), match.group(2)
        if exists(a + b) and not (exists(a) and exists(b)):
            return a + b
        return a + " " + b

    line = APOSTROPHE_INSIDE.sub(join_or_split, line)
    return APOSTROPHE_AT_EDGE.sub("", line)

def remove_stray_dots(line, counts):
    def keep_or_drop(match):
        if is_abbreviation(match.group(1), counts):
            return match.group(0)
        return match.group(1) + " "

    # "S .Pablo" -> "S. Pablo", "pues :el" -> "pues el":
    def drop_or_move(match):
        before = match.group(1)
        if before is None:
            return ""
        if is_abbreviation(before, counts) and "." in match.group(0):
            return before + ". "
        return before + " "

    new = INSIDE.sub(r"\1 \2", line)
    while new != line:
        line = new
        new = INSIDE.sub(r"\1 \2", line)
    line = DOT_THEN_LOWERCASE.sub(keep_or_drop, line)
    return DOTS_AT_START.sub(drop_or_move, line)

counts = count_dots(CLEAN_FOLDER)
changed = 0
for file_name in sorted(os.listdir(CLEAN_FOLDER)):
    path = os.path.join(CLEAN_FOLDER, file_name)
    with open(path, encoding="utf-8") as f:
        lines = f.read().split("\n")
    for k in range(len(lines)):
        if not lines[k].startswith("<pb"):
            new = fix_apostrophes(lines[k], books)
            new = COMMA_INSIDE.sub(", ", new)
            new = QUOTE_INSIDE.sub(" ", new)
            new = remove_stray_dots(new, counts)
            if new != lines[k]:
                changed = changed + 1
            lines[k] = new
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
print(changed, "lines with stray punctuation fixed")


47149 lines with stray punctuation fixed


In [13]:
# Celda 13
# Latin and Italian passages, marked so that modernization and post-correction leave them alone and text analysis
# can leave them out: <foreign xml:lang="la">Dominus regit me, et nihil mihi deerit</foreign>.
# The text is cut into phrases at punctuation, quotes, brackets and tags; each phrase of 3 words or more gets a
# language from lingua (github.com/pemistahl/lingua-py, package lingua-language-detector 2.2.0), choosing only among
# Spanish, Latin and Italian. A phrase is marked if its language is Latin or Italian with confidence 0.9 or more and
# it has a common function word of that language: without that, Spanish in old spelling ("quales", "assi",
# "necessidades") passed for Latin. Neighboring marked phrases are joined. A passage that crosses a line break is
# closed at the end of the line and opened again on the next one, so each line stays whole.
# Quotes of one or two words ("et alii") are left to the modernization, which has its own check for Latin.
# About 1 second per book:
import sys
from collections import Counter
from lingua import Language, LanguageDetectorBuilder

sys.path.insert(0, "ocr-piloto")
import reglas_modernizacion_postcorreccion as reglas  # LAT_STRONG: Latin function words

LANGUAGE_DETECTOR = LanguageDetectorBuilder.from_languages(Language.SPANISH, Language.LATIN, Language.ITALIAN).build()
LANGUAGE_CODES = {Language.LATIN: "la", Language.ITALIAN: "it"}
MIN_CONFIDENCE = 0.9
FUNCTION_WORDS = {"la": reglas.LAT_STRONG,
                  "it": {"il", "che", "gli", "della", "delle", "nel", "nella", "sono", "più", "questo", "quella", "di"}}
PHRASE_BREAK = re.compile(r'<[^>]+>|[.;:?!«»“”"()\[\]]')
FOREIGN_TAG = re.compile(r'</?foreign[^>]*>')


def mark_languages(text):
    text = FOREIGN_TAG.sub("", text)  # Running the cell again starts from scratch
    phrases = []
    start = 0
    for m in list(PHRASE_BREAK.finditer(text)) + [None]:
        end = m.start() if m else len(text)
        if len(re.findall(r"[^\W\d_]+", text[start:end])) >= 3:
            phrases.append((start, end))
        start = m.end() if m else end
    values = LANGUAGE_DETECTOR.compute_language_confidence_values_in_parallel(
        [text[a:b].replace("\n", " ") for a, b in phrases])
    spans = []  # (start, end, code)
    for (a, b), confidences in zip(phrases, values):
        best = confidences[0]
        code = LANGUAGE_CODES.get(best.language)
        words = {w.lower() for w in re.findall(r"[^\W\d_]+", text[a:b])}
        if code and best.value >= MIN_CONFIDENCE and words & FUNCTION_WORDS[code]:
            while text[a].isspace():
                a += 1
            while text[b - 1].isspace():
                b -= 1
            gap = text[spans[-1][1]:a] if spans else None
            if spans and spans[-1][2] == code and re.fullmatch(r'[\s.,;:?!«»“”"()\[\]]*', gap):
                spans[-1] = (spans[-1][0], b, code)  # One passage of several phrases
            else:
                spans.append((a, b, code))
    for a, b, code in reversed(spans):
        pieces = [f'<foreign xml:lang="{code}">{piece}</foreign>' if piece.strip() else piece
                  for piece in text[a:b].split("\n")]
        text = text[:a] + "\n".join(pieces) + text[b:]
    return text, len(spans)


marked = Counter()
for file_name in sorted(os.listdir(CLEAN_FOLDER)):
    path = os.path.join(CLEAN_FOLDER, file_name)
    with open(path, encoding="utf-8") as f:
        text = f.read()
    text, count = mark_languages(text)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
    marked[file_name] = count
print(sum(marked.values()), "Latin or Italian passages marked; most:", marked.most_common(5))


10579 Latin or Italian passages marked; most: [('cabrera-consideraciones-sobre-los-evangelios-de-la-cuaresma.txt', 2051), ('rodriguez-ejercicio-de-perfeccion-y-virtudes-cristianas-tomo2.txt', 935), ('cabrera-consideraciones-sobre-los-evangelios-de-adviento.txt', 851), ('rodriguez-ejercicio-de-perfeccion-y-virtudes-cristianas-tomo1.txt', 804), ('rodriguez-ejercicio-de-perfeccion-y-virtudes-cristianas-tomo3.txt', 643)]


In [14]:
# Celda 14
# Material of the 19th- and 20th-century editors and later additions, removed from the clean texts.
# Cells 15 to 19 run in this order after Celda 13. Every removal is written to REMOVED_LOG (book, page, rule, text),
# so that it can be checked and undone.
EDITOR_FOLDER = "ocr-piloto"
REMOVED_LOG = "ocr-piloto/registro-material-quitado.csv"
removed = []  # Rows of REMOVED_LOG: [book, page, rule, text]

def read_lines(book):
    with open(os.path.join(CLEAN_FOLDER, book + ".txt"), encoding="utf-8") as f:
        return f.read().split("\n")

def write_lines(book, lines):
    with open(os.path.join(CLEAN_FOLDER, book + ".txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

def page_of(lines, index):
    # Page of a line: the last <pb n="N"/> above it
    for k in range(index, -1, -1):
        m = re.match(r'<pb n="(\d+)"/>', lines[k])
        if m:
            return int(m.group(1))
    return 0

def plain(text):
    return re.sub(r"<[^>]+>", "", text)

def balance_small(lines):
    # After cutting lines, a <small> may be left open or a </small> without its <small>: drop the loose tags
    inside = False
    for k, line in enumerate(lines):
        if line.startswith("<pb") or line.startswith("<head>"):
            continue
        opens, closes = line.count("<small>"), line.count("</small>")
        if closes and not inside and not opens:
            line = line.replace("</small>", "")
            closes = 0
        if opens and not closes:
            inside = True
        elif closes:
            inside = False
        lines[k] = line
    if inside:
        last = max(k for k, l in enumerate(lines) if l and not l.startswith("<pb"))
        lines[last] += "</small>"
    return lines

In [15]:
# Celda 15
# Indexes of the edition ("Índice de las cosas más notables"), which the 1860 and 1862 editions print after the text.
# Each one is cut from its first line to its last line, both found by their text:
EDITION_INDEXES = [
    ("agreda-mistica-ciudad-de-dios-tomo3", "<head>ÍNDICE</head>", "FIN DE LA TABLA"),
    ("agreda-mistica-ciudad-de-dios-tomo6", "<head>ÍNDICE</head>", "FIN DE LA TABLA"),
    # Up to the poems that follow the index (p. 407), which stay:
    ("nieremberg-diferencia-entre-lo-temporal-y-eterno", "<head>ÍNDICE</head>", '<pb n="407"/>'),
]
for book, start_text, end_text in EDITION_INDEXES:
    lines = read_lines(book)
    starts = [k for k, l in enumerate(lines) if l.startswith(start_text)]
    if not starts:
        print("not found (already removed?):", book)
        continue
    start = starts[0]
    end = next(k for k in range(start, len(lines)) if end_text in lines[k])
    if end_text.startswith("<pb"):
        end -= 1  # The page break of the next page stays
    removed.append([book, page_of(lines, start), "índice de la edición", f"{end - start + 1} líneas"])
    lines[start:end + 1] = []
    write_lines(book, balance_small(lines))
    print(book, "-", end - start + 1, "lines")

# Closing lines of a volume, which depend on how the edition splits the work:
# "<head>FIN DEL TOMO PRIMERO.</head>", "FIN DE ESTE PRIMER TOMO.", "FIN DEL TOMO Ill" (OCR), "FIN DEL MES DE JUNIO.".
# Other closing lines ("FIN DE LA PRIMERA PARTE.", "LAUS DEO") stay, since they can be in the early editions:
VOLUME_END = re.compile(r"^\W*FIN\s+(DEL|DE\s+ESTE)\s+(\w+\s+)?(TOMO|MES)\b", re.I)
ends = 0
for file_name in sorted(os.listdir(CLEAN_FOLDER)):
    book = file_name[:-4]
    lines = read_lines(book)
    kept = []
    for k, line in enumerate(lines):
        if VOLUME_END.match(plain(line)):
            removed.append([book, page_of(lines, k), "fin de tomo", plain(line)])
            ends = ends + 1
        else:
            kept.append(line)
    if len(kept) < len(lines):
        write_lines(book, balance_small(kept))
print(ends, "volume-end lines")


agreda-mistica-ciudad-de-dios-tomo3 - 2704 lines
agreda-mistica-ciudad-de-dios-tomo6 - 6359 lines
nieremberg-diferencia-entre-lo-temporal-y-eterno - 969 lines


15 volume-end lines


In [16]:
# Celda 16
# Pages scanned twice (or three times) in the same book: Mística Ciudad de Dios t. 4 pp. 156-217, Godínez, Nieremberg's
# Catecismo, Ochoa's Tesoro... Two pages are the same if 70% or more of the 8-word sequences of the shorter one are in
# the other one (pages of 60 words or more). The first appearance stays; the later ones are emptied. Every pair
# checked by eye was a real repetition, with only OCR differences.
def page_spans(lines):
    # {page number: (index of its <pb> line, index of the next <pb> line)}
    starts = [k for k, l in enumerate(lines) if l.startswith("<pb")]
    ends = starts[1:] + [len(lines)]
    return {int(re.search(r"\d+", lines[a]).group()): (a, b) for a, b in zip(starts, ends)}

def shingles(text):
    w = re.findall(r"\w+", plain(text).lower())
    return set(" ".join(w[i:i + 8]) for i in range(len(w) - 7)) if len(w) >= 60 else set()

for file_name in sorted(os.listdir(CLEAN_FOLDER)):
    book = file_name[:-4]
    lines = read_lines(book)
    spans = page_spans(lines)
    sets = {p: shingles(" ".join(lines[a:b])) for p, (a, b) in spans.items()}
    repeated = set()
    for p in sorted(sets):
        if not sets[p] or p in repeated:
            continue
        for q in sorted(sets):
            if q <= p or not sets[q] or q in repeated:
                continue
            if len(sets[p] & sets[q]) / min(len(sets[p]), len(sets[q])) >= 0.7:
                repeated.add(q)
    if not repeated:
        continue
    for q in sorted(repeated, reverse=True):
        a, b = spans[q]
        removed.append([book, q, "página repetida", " ".join(lines[a + 1:b])])
        lines[a + 1:b] = []  # The page break stays, empty: the PDF page numbers stay valid
    write_lines(book, balance_small(lines))
    print(book, "- repeated pages:", sorted(repeated))

agreda-mistica-ciudad-de-dios-tomo2 - repeated pages: [140, 141]


agreda-mistica-ciudad-de-dios-tomo4 - repeated pages: [186, 187, 188, 189, 190, 192, 193, 194, 196, 197, 198, 199, 200, 201, 203, 204, 205, 208, 209, 210, 211, 212, 213, 214, 216, 217]


godinez-practica-de-la-teologia-mistica - repeated pages: [58, 59, 88, 89, 140, 141, 149, 191, 218, 219, 264, 265, 324, 325, 336, 337, 366, 367, 418, 419]


loyola-ejercicios-espirituales - repeated pages: [74]


luis-de-la-puente-meditaciones-espirituales-tomo1 - repeated pages: [383]


luis-de-la-puente-meditaciones-espirituales-tomo2 - repeated pages: [260]


... (3 outputs left out)

nieremberg-practica-del-catecismo-romano - repeated pages: [32, 33, 50, 51, 84, 85, 192, 193]


nieremberg-varones-ilustres-de-la-compania-de-jesus-tomo6 - repeated pages: [602, 606, 607, 608]


nieremberg-varones-ilustres-de-la-compania-de-jesus-tomo9 - repeated pages: [468, 469, 470, 471, 472, 473]


yepes-vida-de-santa-teresa-de-jesus - repeated pages: [70, 120, 121, 122, 123, 142, 144, 146]


In [17]:
# Celda 17
# Small-type blocks (<small>) that are not the author's text:
# 1. In every book, OCR noise: a block with fewer than 15 letters ("mo, tea!", "| ; | !") and no year, month,
#    JESUS or word of 5 letters or more. In the hand-labeled sample
#    this takes 31 of the 34 noise blocks and 18 editor's notes, and 4 short scraps of the author's text.
# 2. Juan de los Ángeles, Obras místicas (Madrid 1912): the editor's margin notes ("La mayor de las ganancias es ganar
#    una alma para Dios.") are blocks of 15 words or fewer; footnotes start with "(*)" or "(1)".
# 3. Silverio's editions (Teresa de Jesús, Burgos 1915-1919; Juan de la Cruz, Burgos 1929): numbered footnotes
#    ("1 Así lo escribe la Santa...") and the critical apparatus ("Bg. y P: unción.", "E p. omite...", "Borrado: ...").
#    Quotations and titles in small type ("Romance 9. Del nacimiento.") stay.
# 4. Ribera, Vida de Santa Teresa (1908): the editor's footnotes, page by page (see below).
# 5. Fray Luis de León, De los nombres de Cristo (Madrid s. a.): the summaries of each name (see below).
# 6. Sancha, Romancero y cancionero sagrados: the editor's source after each poem (see SANCHA_SOURCE).
APPARATUS = re.compile(r"^\W{0,3}\d{1,2}\s+\S|^\W{0,3}([A-Z][a-zA-Z]{0,2}\.?|S M|E p\.)(\s?(y|,)\s?[A-Z][a-zA-Z]{0,2}\.?)*\s?(:|omite|añade)\s"
                       r"|Borrad|autógrafo|copista|tach[óa]|ediciones|Véase la nota|Falta este verso|Por distracción|había escrito")
NOTE_MARK = re.compile(r"^\W{0,2}\(\s?(\*|\d{1,2}|\?)\s?\)")
# Short blocks that are not noise: dates of letters ("Madrid, 11 de Abril 1659."), "JESUS", short titles
# ("REMEDIO 11.", "FRUTO"): a year, a month, JESUS or JHS, or any word of 5 letters or more.
KEEP_SHORT = re.compile(r"1[5-9]\d\d|enero|febrero|marzo|abril|mayo|junio|julio|agosto|setiembre|septiembre|octubre|"
                        r"noviembre|diciembre|jes[uú]s|j\.?h\.?s|[^\W\d_]{5,}", re.I)
LEFT_OVER = "@@QUITADO@@"  # Marks the place of a removed block until the lines are rebuilt

# In these books of Silverio's editions every small-type block left is apparatus or an editor's note (checked by eye
# in the manual review); in the others (Conceptos, the poems of Juan de la Cruz) the quotations and
# titles in small type stay:
# Editions of Silverio de Santa Teresa (Burgos, 1915-1929):
SILVERIO_BOOKS = [
    'juan-de-la-cruz-cartas',
    'juan-de-la-cruz-llama-de-amor-viva-primera-redaccion',
    'juan-de-la-cruz-llama-de-amor-viva-segunda-redaccion',
    'juan-de-la-cruz-poesias-y-consejos',
    'teresa-de-jesus-conceptos-del-amor-de-dios',
    'teresa-de-jesus-constituciones',
    'teresa-de-jesus-exclamaciones-del-alma-a-dios',
    'teresa-de-jesus-libro-de-la-vida',
    'teresa-de-jesus-libro-de-las-fundaciones',
    'teresa-de-jesus-modo-de-visitar-los-conventos',
    'teresa-de-jesus-relaciones-espirituales',
]
SILVERIO_ALL_SMALL = [
    'juan-de-la-cruz-llama-de-amor-viva-primera-redaccion',
    'juan-de-la-cruz-llama-de-amor-viva-segunda-redaccion',
    'teresa-de-jesus-relaciones-espirituales',
    'teresa-de-jesus-libro-de-las-fundaciones']

# Sancha, Romancero y cancionero sagrados (BAE): after each poem the 1855 editor gives its author and source in small
# type ("LOPE DE VEGA CARPIO.— Coleccion de obras sueltas.— Madrid, 1778, en 4.º"; "EL MISMO."). Reviewed by eye:
# the games ("El juego de..."), emblems ("Pintóse...") and verses in small type stay; a few blocks that mix a
# verse with the source lose the verse.
SANCHA_SOURCE = re.compile(r"[A-ZÁÉÍÓÚ][\wÁÉÍÓÚáéíóú]+\.?\s?—|—\s?[A-ZÁÉÍÓÚ]|1[4-8]\d\d|manuscrit|[Tt]omo|pág|impres|edici[oó]n|hojas"
                           r"|en (4|8|12|fol)|autor|Madrid|Sevilla|Valencia|Zaragoza|Alcal|Toledo|Lisboa|Salamanca|Valladol"
                           r"|Barcelona|Lóndres|^\W{0,2}\*\s?\w+\. |etc\.\s*\W{0,6}$|anterior|^\W{0,3}E[lnsa]\s?m\w{2,4}o"
                           r"|Soneto en eco|intitulado|almirante de Castilla|desires muy sotiles|montes y breñas"
                           r"|las obras hechas á la beat|al alma compuso|^\W{0,3}San JUAN DE LA", re.M)
SANCHA_KEEP = re.compile(r"^\W{0,3}(El juego|P[ií]nt[oó]se|—\s?(Con ellos|En penas)|Y de los pecados|En Valladoli|AGUST|CUELLO)")

def editor_block(book, text):
    t = plain(text).strip()
    if len(re.findall(r"[^\W\d_]", t)) < 15 and not KEEP_SHORT.search(t):
        return "ruido de OCR"
    if book.startswith("juan-de-los-angeles") and (len(t.split()) <= 15 or NOTE_MARK.match(t)):
        return "apostilla o nota del editor"
    if book == "varios-romancero-y-cancionero-sagrados" and SANCHA_SOURCE.search(t) and not SANCHA_KEEP.match(t):
        return "fuente indicada por el editor"
    if book in SILVERIO_ALL_SMALL or (book in SILVERIO_BOOKS and APPARATUS.search(t)):
        return "nota o aparato crítico del editor"
    return None

for file_name in sorted(os.listdir(CLEAN_FOLDER)):
    book = file_name[:-4]
    text = "\n".join(read_lines(book))
    def drop(m):
        rule = editor_block(book, m.group(0))
        if not rule:
            return m.group(0)
        pages_before = re.findall(r'<pb n="(\d+)"/>', text[:m.start()])
        removed.append([book, int(pages_before[-1]) if pages_before else 0, rule, plain(m.group(0))])
        # The page breaks inside the block stay
        return "\n".join(re.findall(r'<pb n="\d+"/>', m.group(0)) + [LEFT_OVER])
    new_text = re.sub(r"<small>.*?</small>", drop, text, flags=re.S)
    if new_text != text:
        # A line that held only the block goes; a block in the middle of a line leaves the rest of the line
        lines = [l.replace(LEFT_OVER, "").rstrip() for l in new_text.split("\n") if l.strip() != LEFT_OVER]
        write_lines(book, lines)
# 4. Ribera, Vida de Santa Teresa (Barcelona 1908, ed. Jaime Pons): Ribera paraphrases Teresa, and the editor's footnotes
#    quote her own words at length (Vida, letters, poems), with notes on places and people. The notes fill the bottom of
#    the page, but the OCR reads some of their lines in normal type, so everything from the first small-type line of a
#    page to the end of the page goes (checked by eye for the 58 pages with notes). On p. 376 the OCR
#    mixed the text with the note line by line, and some broken half-lines of Ribera go with it.
#    The note calls in the text ("sus enfermedades (1),") go too.
RIBERA = "ribera-vida-de-santa-teresa-de-jesus"
lines = read_lines(RIBERA)
page_starts = [k for k, l in enumerate(lines) if l.startswith("<pb")] + [len(lines)]
for a, b in reversed(list(zip(page_starts, page_starts[1:]))):
    first = next((k for k in range(a, b) if "<small>" in lines[k]), None)
    if first is None:
        continue
    removed.append([RIBERA, page_of(lines, a), "nota del editor", " ".join(plain(l) for l in lines[first:b])])
    lines[first:b] = []
calls = sum(len(re.findall(r"\s?\((?:\d{1,2}|\*)\)", l)) for l in lines)
lines = [re.sub(r"\s?\((?:\d{1,2}|\*)\)", "", l) for l in lines]
removed.append([RIBERA, 0, "llamadas de nota del editor", f"{calls} llamadas"])
write_lines(RIBERA, balance_small(lines))
print(RIBERA, "-", calls, "note calls")
# 5. Fray Luis de León, De los nombres de Cristo (Madrid s. a.): the summary at the head of each
#    name and introduction ("Es llamado Cristo Pimpollo, y explícase cómo le conviene este nombre..."). They are not in
#    the 1595 edition (Salamanca, Juan Fernández; HathiTrust uc1.31822035057827), and the corpus follows the early
#    editions. Each is found by its first words (the OCR reads some of them in normal type) and
#    cut up to the end of its sentence, at most 4 lines.
NOMBRES_BOOKS = ["luis-de-leon-de-los-nombres-de-cristo"]
NOMBRES_SUMMARIES = [
    "Dase razon y motivo de la obra", "Introducese en el asunto con la idea de un coloquio",
    "Explicase que viene a ser nombre", "Es llamado Cristo pimpollo", "Declarase como Cristo tiene el nombre de faces",
    "Es Cristo llamado Camino", "Llamase Cristo Pastor", "Se le da a Cristo el nombre de Monte",
    "Llamase Cristo Padre del siglo futuro", "Descripcion de la miseria humana", "De como se llama Cristo Brazo de Dios",
    "Es Cristo llamado Rey", "Explicase que cosa es paz", "Llamase Cristo Esposo", "Se da solucion a algunos reparos",
    "Cuan propiamente se llama Cristo Hijo de Dios", "Tratase del nombre el Amado",
    "Que significa y como le conviene solo a Cristo el nombre de Jesus", "De como Cristo es llamado Cordero",
]
from rapidfuzz import fuzz
import unicodedata

def simple(text):
    text = unicodedata.normalize("NFD", plain(text).lower())
    return " ".join(re.findall(r"[a-zñ]+", "".join(c for c in text if not unicodedata.combining(c))))

for book in NOMBRES_BOOKS:
    lines = read_lines(book)
    found = 0
    for summary in NOMBRES_SUMMARIES:
        target = simple(summary)
        best = max(range(len(lines)), key=lambda k: fuzz.ratio(simple(lines[k])[:len(target)], target))
        if fuzz.ratio(simple(lines[best])[:len(target)], target) < 80:
            print("not found:", book, "-", summary)
            continue
        # Up to the line that ends the sentence. A line that does not end in a word ('este "nombre, > 7'), followed by
        # one that starts in capitals, is already followed by the author's text ("Llámase también Camino...")
        end = best
        while (end < best + 3 and not re.search(r"[.]\W*(</small>)?\W*$", lines[end])
               and not (re.search(r"[^\w\s]\W*\d?\W*$", plain(lines[end]))
                        and re.match(r"\W*[A-ZÁÉÍÓÚÑ«—]", plain(lines[end + 1])))):
            end = end + 1
        removed.append([book, page_of(lines, best), "resumen del editor", " ".join(plain(l) for l in lines[best:end + 1])])
        lines[best:end + 1] = [l for l in lines[best:end + 1] if l.startswith("<pb")]
        found = found + 1
    write_lines(book, balance_small(lines))
    print(book, "-", found, "summaries")
from collections import Counter
print(Counter(r[2] for r in removed))

ribera-vida-de-santa-teresa-de-jesus - 396 note calls


luis-de-leon-de-los-nombres-de-cristo - 19 summaries
Counter({'ruido de OCR': 545, 'apostilla o nota del editor': 310, 'fuente indicada por el editor': 115, 'página repetida': 85, 'nota o aparato crítico del editor': 58, 'nota del editor': 58, 'resumen del editor': 19, 'fin de tomo': 15, 'índice de la edición': 3, 'llamadas de nota del editor': 1})


In [18]:
# Celda 18
# Material of the 19th- and 20th-century editors and later additions, listed in ocr-piloto/resumenes-del-editor.csv
# with their lines in the clean text: chapter summaries, summaries of letters, editor's notes, and the passages that
# are not in the early editions used for checking (Ripalda against Ochoa de Capanaga 1656).
# Part of the first and last line can stay (the end of the chapter title, "JESUS"...).
# The line numbers of the list drift (rows written at different stages of the cleaning), so each row is found by its
# words: a line holding its first 5 words and, below it, a line holding its last 5 words, as close as possible to the
# row's own number of lines and to the shift of the row below it. A row not found is skipped and counted.
from collections import defaultdict

def words(text):
    return re.findall(r"\w+", plain(text).lower())

summaries = {}
with open(os.path.join(EDITOR_FOLDER, "resumenes-del-editor.csv"), encoding="utf-8") as f:
    for row in csv.DictReader(f):
        if os.path.exists(os.path.join(CLEAN_FOLDER, row["libro"] + ".txt")):
            summaries.setdefault(row["libro"], []).append(row)

skipped = []
for book, rows in summaries.items():
    lines = read_lines(book)
    # Every word of the book, its line, and where each word appears. Removed lines are only marked until the end, so
    # that the indexes stay valid:
    tokens, line_of = [], []
    for k, line in enumerate(lines):
        if not line.startswith("<pb"):
            for w in words(line):
                tokens.append(w)
                line_of.append(k)
    where = defaultdict(list)
    for i, w in enumerate(tokens):
        where[w].append(i)
    alive = [True] * len(lines)
    replacement = {}  # first removed line -> lines that take the place of the passage

    def find_sequence(sequence):
        n = len(sequence)
        return [i for i in where.get(sequence[0], []) if tokens[i:i + n] == sequence and alive[line_of[i]]
                and alive[line_of[i + n - 1]]]

    shift = 0
    # From the bottom up, as the rows were written
    for row in sorted(rows, key=lambda r: -int(r["linea_desde"])):
        row_words = words(row["texto"])
        if not row_words:
            skipped.append([book, row["linea_desde"], "(sin palabras)"])
            continue
        start_words, end_words = row_words[:5], row_words[-5:]
        size = int(row["linea_hasta"]) - int(row["linea_desde"]) + 1
        expected = int(row["linea_desde"]) - 1 + shift
        ends = find_sequence(end_words)
        best = None
        for s in find_sequence(start_words):
            first = line_of[s]
            # The nearest end at or after the start
            later = [e for e in ends if e + len(end_words) >= s + len(start_words)]
            if not later:
                continue
            last = line_of[later[0] + len(end_words) - 1]
            # Not more than twice the expected lines plus 40, and nothing removed inside
            if last - first + 1 > 2 * size + 40 or not all(alive[first:last + 1]):
                continue
            score = (abs(first - expected) > 300, abs(first - expected), abs(last - first + 1 - size))
            if best is None or score < best[0]:
                best = (score, first, last)
        if best is None:
            skipped.append([book, row["linea_desde"], " ".join(row_words[:8])])
            continue
        _, first, last = best
        shift = first - (int(row["linea_desde"]) - 1)
        kept = [row[c] for c in ("conservar_al_inicio", "conservar_al_final") if row[c]]
        page_breaks = [l for l in lines[first:last + 1] if l.startswith("<pb")]
        removed.append([book, page_of(lines, first), row["tipo"] or "resumen del editor", row["texto"]])
        for k in range(first, last + 1):
            alive[k] = False
        replacement[first] = kept + page_breaks
    new_lines = []
    for k, line in enumerate(lines):
        if k in replacement:
            new_lines.extend(replacement[k])
        if alive[k]:
            new_lines.append(line)
    write_lines(book, balance_small(new_lines))
print(sum(len(r) for r in summaries.values()) - len(skipped), "passages removed;", len(skipped), "not found")
for s in skipped:
    print("  not found:", *s)

555 passages removed; 0 not found


In [19]:
# Celda 19
# The log of everything removed by cells 15 to 18:
with open(REMOVED_LOG, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["libro", "pagina", "regla", "texto"])
    writer.writerows(removed)
print(len(removed), "rows in", REMOVED_LOG)

1764 rows in ocr-piloto/registro-material-quitado.csv
